# Prompt Engineering Patterns

Effective prompting is essential for getting the best out of LLMs. This notebook covers:
1. **Zero-shot** prompting
2. **Few-shot** in-context learning
3. **Chain-of-thought** (CoT) reasoning
4. **Structured output** formatting
5. **Evaluation** of prompt strategies

In [ ]:
import numpy as np
import json
from typing import Dict, List

# We simulate LLM responses for reproducibility.
# In practice, replace SimulatedLLM with an API call (OpenAI, Anthropic, etc.)

class SimulatedLLM:
    """A rule-based simulator for demonstrating prompt patterns."""
    def __init__(self):
        self.call_count = 0
    
    def generate(self, prompt: str, max_tokens: int = 200) -> str:
        self.call_count += 1
        prompt_lower = prompt.lower()
        
        # Sentiment analysis
        if 'sentiment' in prompt_lower:
            if any(w in prompt_lower for w in ['amazing', 'great', 'love', 'excellent', 'fantastic']):
                return 'Positive'
            elif any(w in prompt_lower for w in ['terrible', 'awful', 'hate', 'worst', 'horrible']):
                return 'Negative'
            return 'Neutral'
        
        # Math with chain-of-thought
        if 'step by step' in prompt_lower and ('total' in prompt_lower or 'cost' in prompt_lower):
            return ('Step 1: Identify the quantities.\n'
                    'Step 2: Compute intermediate values.\n'
                    'Step 3: Sum to get the total.\n'
                    'Answer: 42')
        
        # Classification
        if 'classify' in prompt_lower or 'category' in prompt_lower:
            if 'python' in prompt_lower or 'code' in prompt_lower:
                return 'Technology'
            elif 'election' in prompt_lower or 'government' in prompt_lower:
                return 'Politics'
            return 'General'
        
        # JSON output
        if 'json' in prompt_lower:
            return json.dumps({'result': 'extracted_entity', 'confidence': 0.92})
        
        return 'I can help with that task.'

llm = SimulatedLLM()
print('Simulated LLM ready. Replace with real API for production use.')

## 1. Zero-Shot Prompting

Give the model a task description with **no examples**. Works well for simple, well-defined tasks.

In [ ]:
# Zero-shot sentiment analysis
def zero_shot_sentiment(text: str) -> str:
    prompt = f"""Classify the sentiment of the following text as Positive, Negative, or Neutral.

Text: \"{text}\"
Sentiment:"""
    return llm.generate(prompt)

test_texts = [
    "This product is amazing, I love it!",
    "The service was terrible and the food was awful.",
    "The package arrived on Tuesday.",
]

print('Zero-shot Sentiment Analysis:')
for text in test_texts:
    result = zero_shot_sentiment(text)
    print(f'  "{text[:50]}..." -> {result}')

## 2. Few-Shot Prompting

Provide **examples** in the prompt to guide the model's behaviour.
This leverages in-context learning -- no gradient updates needed.

In [ ]:
def few_shot_classify(text: str) -> str:
    prompt = f"""Classify the following text into a category.

Examples:
Text: "The new Python 3.12 release includes performance improvements."
Category: Technology

Text: "The election results showed a narrow margin of victory."
Category: Politics

Text: "The team won the championship in overtime."
Category: Sports

Text: \"{text}\"
Category:"""
    return llm.generate(prompt)

test_articles = [
    "A new Python library for code generation was released today.",
    "The government announced new fiscal policy measures.",
]

print('Few-shot Classification:')
for text in test_articles:
    result = few_shot_classify(text)
    print(f'  "{text[:60]}..." -> {result}')

## 3. Chain-of-Thought (CoT) Prompting

Ask the model to **reason step by step**. This dramatically improves performance on
tasks requiring multi-step reasoning (arithmetic, logic, planning).

In [ ]:
# Without CoT
prompt_no_cot = """Q: A store sells 3 shirts at $25 each and 2 pants at $40 each. What is the total cost?
A:"""

# With CoT
prompt_cot = """Q: A store sells 3 shirts at $25 each and 2 pants at $40 each. What is the total cost?
Let's think step by step.
A:"""

print('Without Chain-of-Thought:')
print(f'  {llm.generate(prompt_no_cot)}')

print('\nWith Chain-of-Thought:')
print(f'  {llm.generate(prompt_cot)}')

print('\n--- Prompt Design Pattern ---')
print('Adding "Let\'s think step by step" triggers explicit reasoning.')
print('This is especially useful for math, logic, and planning tasks.')

## 4. Structured Output (JSON Mode)

Force the model to return structured data by specifying the output format explicitly.

In [ ]:
def extract_entities(text: str) -> dict:
    prompt = f"""Extract named entities from the following text.
Return ONLY valid JSON with keys: "persons", "organizations", "locations".

Text: \"{text}\"

JSON output:"""
    response = llm.generate(prompt)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {'error': 'Failed to parse JSON', 'raw': response}

result = extract_entities("Dr. Smith from MIT visited Paris for the AI conference.")
print('Structured extraction result:')
print(json.dumps(result, indent=2))

print('\n--- Best Practices ---')
print('1. Specify the exact JSON schema in the prompt.')
print('2. Say "Return ONLY valid JSON" to reduce preamble.')
print('3. Always wrap with try/except for robustness.')

In [ ]:
# Prompt template system for reusable prompts
class PromptTemplate:
    def __init__(self, template: str):
        self.template = template
    
    def format(self, **kwargs) -> str:
        return self.template.format(**kwargs)

# Example: reusable sentiment template
sentiment_template = PromptTemplate(
    template="""You are a sentiment analysis expert.
Analyse the sentiment of the following {language} text.
Return one of: Positive, Negative, Neutral.

Text: \"{text}\"
Sentiment:"""
)

prompt = sentiment_template.format(language='English', text='This is a fantastic course!')
print('Generated prompt:')
print(prompt)
print(f'\nResult: {llm.generate(prompt)}')
print(f'\nTotal LLM calls in this notebook: {llm.call_count}')

# --- 5. Prompt Template System ---
# Best practices summary:
#   Zero-shot      -> Simple tasks, clear instruction
#   Few-shot       -> Ambiguous/domain tasks, 3-5 diverse examples
#   Chain-of-thought -> Multi-step reasoning, "Let's think step by step"
#   Structured output -> Downstream parsing, specify exact schema
#   Role prompting -> Persona/expertise, "You are an expert..."
#   Self-consistency -> High-stakes, sample N times + majority vote

class PromptTemplate:
    def __init__(self, template: str):
        self.template = template
    
    def format(self, **kwargs) -> str:
        return self.template.format(**kwargs)

# Example: reusable sentiment template
sentiment_template = PromptTemplate(
    template="""You are a sentiment analysis expert.
Analyse the sentiment of the following {language} text.
Return one of: Positive, Negative, Neutral.

Text: \"{text}\"
Sentiment:"""
)

prompt = sentiment_template.format(language='English', text='This is a fantastic course!')
print('Generated prompt:')
print(prompt)
print(f'\nResult: {llm.generate(prompt)}')
print(f'\nTotal LLM calls in this notebook: {llm.call_count}')